# More Magnetism

## 1. Spin Hamiltonians 

<p align="center"><em>...magnets, how do they work?</em></p>
<p align="center">(Insane Clown Posse, 2009)</p>

One way to answer the question above is by writing a spin Hamiltonian:

$$
\mathcal{H} =
-\sum_{i<j} J_{ij}\,\mathbf{S}_i\cdot\mathbf{S}_j
-\sum_{i<j} \mathbf{D}_{ij}\cdot(\mathbf{S}_i\times\mathbf{S}_j)
-\sum_i k_i^{u}\,(\mathbf{S}_i\cdot\hat{\mathbf{e}}_i)^2
-\sum_i \mu_i\,\mathbf{S}_i\cdot\mathbf{B}_{\mathrm{app}}
+\mathcal{H}_{\mathrm{dip}}
+\mathcal{H}_{\mathrm{cubic}}
+\mathcal{H}_{\mathrm{magnetoelastic}}
$$

Here, $J_{ij}$ is the exchange coupling between atoms $i$ and $j$, $\mathbf{D}_{ij}$ is the Dzyaloshinskii-Moriya (DM) vector, $k_i^u$ is the easy-axis anisotropy constant, $\mu_i$ is the magnetic moment, and $\mathbf{B}_{\mathrm{app}}$ is the applied field. The remaining terms represent dipolar interactions, cubic anisotropy, and magnetoelastic coupling.

In simple terms: exchange favors collinearity (parallel for $J_{ij}>0$, antiparallel for $J_{ij}<0$), DM favors chiral canting, anisotropy favors specific crystal directions, and the Zeeman term aligns moments with the external field. For a two-spin exchange + DM model, the canting angle is given by $\tan\theta = D/J$.

To model magnetic properties with this Hamiltonian, we typically:
1. Select material parameters, temperature, and applied field.
2. Build a spin array representing the material and assign initial spin orientations.
3. Equilibrate the system using Metropolis Monte Carlo (MC).

This is roughly how the _Vampire_ program works. Its MC routine is:

<div style="border: 1px solid #b7c3d0; border-radius: 8px; padding: 14px 14px; background: #f8fafc;">
<strong><em>Vampire</em> algorithm</strong>

1. <strong>Start:</strong> compute the energy of the initial configuration $E_{\mathrm{old}}$.
2. <strong>Repeat for</strong> $n$ <strong>steps:</strong>
   1. Propose a trial spin move to generate a new configuration.
   2. Compute the energy change $\Delta E = E_{\mathrm{new}}-E_{\mathrm{old}}$, then:
      1. If $\Delta E\le 0$, accept the move.
      2. If $\Delta E>0$, accept with Metropolis probability $p=\exp(-\Delta E/k_B T)$; otherwise reject.
3. <strong>End.</strong>
</div>

So, all we have to do to begin is determine the parameters in the equation above. 

## 2. NiO

### 2.1. Introduction

Nickel oxide (NiO) is a prototypical transition-metal oxide. It is an antiferromagnet: spins are ferromagnetically aligned within $(111)$ planes and alternate in direction between neighboring planes. Its magnetic order is dominated by superexchange (through-bond) interactions, which make its Néel temperature – the temperature at which thermal fluctuations overcome magnetic ordering – unusually high ($T_\mathrm{N}\approx 523\,\mathrm{K}$). As NiO is a wide-gap insulator, it is useful for studying spin transport and room-temperature antiferromagnetic spintronics in general.

### 2.2. NiO Electronic Structure at PBE

NiO is often described as a correlated insulator. Let's find out what that means. Compute the electronic structure and band gap of NiO.

In [ ]:
"""
Remember, if you change something here you 
need to restart the kernel and execute
this cell again to import the changes. 
"""
from qe_funs import *

In [ ]:
# inspect the geometry
write_geom("scf.in")

# run an SCF calculation at 6x6x6, or 8x8x8 if you are brave
!pw.x < scf.pbe.in > scf.pbe.out
# run an NSCF calculation at 8x8x8, or 10x10x10 if you are brave
!pw.x < nscf.pbe.in > nscf.pbe.out
# run a DOS calculation and plot the total DOS
# !dos.x < dos.pbe.in > dos.pbe.out
# !mv work.dos work.dos.pbe
plot_dos(file_name="work.dos.pbe")
# calculate the projected DOS
!projwfc.x < projwfc.pbe.in > projwfc.pbe.out


In [ ]:
# numbers numbers
e_range = # set range here 
e_fermi = # find the fermi level 

# make the figure 
fig, ax = plt.subplots(1, 1)

# load and plot the total DOS
dos = np.genfromtxt(f"pbe.pdos.pdos_tot", skip_header=1)
energy = dos[:, 0] - e_fermi
dos_up = dos[:, 1]
dos_dn = dos[:, 2]
ax.plot(energy, dos_up, color = colours["orange"], label="spin up") 
ax.plot(energy, -dos_dn, color = colours["blue"], label="spin down") 

# here is a pdos, you can add a few of these 
pdos_ni1 = np.genfromtxt(f"really long file name", skip_header=1) 
pdos_ni1_up = pdos_ni1[:, 1]
pdos_ni1_dn = pdos_ni1[:, 2]
ax.plot(energy, pdos_ni1_up, color = colours["orange"], label="3d Ni1 spin up", linestyle='dashed') 
ax.plot(energy, -pdos_ni1_dn, color = colours["blue"], label="3d Ni1 spin down", linestyle='dashed') 

# some formatting
ax.axvline(0, linestyle='dashed', color = "k") # fermi energy as vertical line
ax.axhline(0, color = "k") # horizontal line at zero
ax.set_xlim(-e_range, e_range)
ax.set_ylim(-1.1 * np.max(dos_dn), 1.1 * np.max(dos_up))
ax.set_ylabel("number of states")
ax.tick_params(labelleft=False, left=False)
ax.legend(frameon=False, loc='upper right')
plt.show() # show the figure


 Analyse ```projwfc.pbe.out```. What is the crystal splitting in NiO? 

What is the band gap of NiO at PBE? The experimental value is $E_\mathrm{g} \approx$ 4 eV. This is quite catastrophic. How can we recover from this result? 



## 3. Beyond GGA

As we mentioned in week 2, GGA functionals over-hybridise the orbitals and over-delocalise the electrons. This is usually addressed by adding exact exchange either empirically (hybrid DFT) or rigorously (_GW_). A cheaper alternative is DFT+_U_. 

### 3.1 DFT+_U_

Conceptually, DFT+_U_ adds Hubbard-like term to the Hamiltonian: 

$$H = H_{\mathrm{DFT}} + \sum_{i,l,s} U_{i,l} \ n_{i,l,s} (1 - n_{i,l,s})$$

where $U_{i,l}$ is the energy added according to the occupations of atomic orbital $l$ on atom $i$, which are computed as in the ```projwfc.in``` routine, and $s$ runs over spins. This correction pushes the occupations of the orbitals $\phi_{i,l}$ towards 0 or 1, potentially decreasing their hybridisation. 

The value of $U$ is usually determined empirically, or used as a parameter to explore the possible behaviour. It can be also computed from first principles, but this is not straightforward. DFT+U is popular because it adds very little computational overhead. 

Here is a minimal input for NiO, adding $U$ to the Ni 3d orbitals. 

<div style="border: 1px solid #b7c3d0; border-radius: 8px; padding: 14px 14px; background: #f8fafc;">
<pre style="margin:0; line-height:1.15;">
HUBBARD (ortho-atomic)
U Ni1-3d 6.0
U Ni2-3d 6.0
</pre>
</div>

Armed with this knowledge, now rerun the calculations and plot the density of states again. Also reuse the code above to plot the projected DOS. 

In [ ]:
# run an SCF calculation at 6x6x6, or 8x8x8 if you are brave
!pw.x < scf.pbeU.in > scf.pbeU.out
# run an NSCF calculation at 8x8x8, or 10x10x10 if you are brave
!pw.x < nscf.pbeU.in > nscf.pbeU.out
# run a DOS calculation and plot the total DOS
# !dos.x < dos.pbeU.in > dos.pbeU.out
# !mv work.dos work.dos.pbeU
plot_dos(file_name="work.dosU.pbe")
# calculate the projected DOS
!projwfc.x < projwfc.pbeU.in > projwfc.pbeU.out

Ok, this is better! Now we can be more confident in our results.

### 3.2. Exchange coupling in NiO

All calculations that we performed were **collinear**, which means that spins were oriented in either $+z$ or $–z$. This means that we can use them to estimate the exchange coupling $J_{ij}$. For the _Vampire_ Hamiltonian, $J_{ij}$ is given by: 

$$ J_{ij} = \frac{E_{\mathrm{AFM}} - {E_{\mathrm{FM}}}}{zS^2}$$

In the equation above, $z$ is the number of Ni-Ni neighbours, and $s = 1$ is the spin quantum number of $\mathrm{Ni}^{2+}$. The Vampire Hamiltonian uses classical unit-vector spins, so $S^2 = s^2 = 1$.

In [ ]:
# !pw.x < scfU_FM.in > scfU_FM.out
# e_high = ! grep "total energy       " scfU_FM.out | tail -1 | awk '{print $5}' 
# e_low = ! grep "total energy       " scfU.out | tail -1 | awk '{print $5}' 

J = # you can do it

Now that we have both the $\mu_\mathrm{B}$ and $J_{ij}$, and we can proceed to model our material.

## 4. Néel temperature in NiO

### 4.1. Specific heat 

Experimentally, we can determine the Néel temperature $T_\mathrm{N}$ by looking at the changes in the specific heat capacity $C$.  Below $T_\mathrm{N}$, the system is highly ordered and $C$ is small because only spin-wave-like excitations are available. Around $T_\mathrm{N}$ many competing spin configurations are available, so $C$ is maximal. Above $T_\mathrm{N}$ the system is largely disordered, so $C$ decreases again. $C$ can be computed as:

$$ C = \frac{\langle U^2 \rangle - \langle U \rangle ^2}{kT^2} $$

where $U$ is the total energy of the system, $T$ is the temperature, and $k$ is the Boltzmann constant. 

### 4.2. Using _Vampire_

_Vampire_ uses at least two input files. The first is always called ```input```, and it contains most of the keywords, including a specification of which materials are modelled. The spin Hamiltonian parameters of these materials are given in the ```.mat``` file. 

_Vampire_ runs in two modes: **equilibrium** and **spin dynamics**. In the **equilibrium** mode, the MC algorithm outlined in section (1.) is employed and the system is run for many steps, which may be split into equilibration (where we allow the system to relax and don't collect data) and production. We will only run this mode today. In the much more fun **spin dynamics** mode a time-resolved simulation is performed, and the evolution of the system is tracked. 

### 4.3. Your first Vampire calculation

Inspect the current ```input.Co``` and ```Co.mat``` files. Then, run Vampire. This should produce ```output``` and ```log``` files – take a look. 

In [ ]:
!cp input.Co input
!/path/to/vampire-serial
!cp output output.Co
!cp log log.Co

### 4.4. Computing the Néel temperature of NiO

To estimate the Néel temperature of NiO, we shall run _Vampire_ in ```curie-temperature``` mode. Inspect ```input.NiO``` and ```NiO.mat``` and fill in the missing keywords. Then, estimate the Néel temperature of NiO.

In [ ]:
!cp input.NiO input
!/path/to/vampire-serial
!cp output output.NiO
!cp log log.NiO

### 4.5. Curie temperature of Co

Now it's your turn to calculate the Curie temperature of Co. Good luck! 

## 5. Magnetic field required for a spin-flip

Sometimes it is possible to change the overall magnetisation of a material by applying a magnetic field. Here we shall look at $\mathrm{CrI}_3$. First, go to  [Materials Project](https://next-gen.materialsproject.org/materials) and take a look at its structure. 

$\mathrm{CrI}_3$ is pretty cool because it has relatively strong FM coupling within a layer, and weaker AFM coupling between layers. This means that magnetisation can be switched "on" and "off" with a magneetic field. 

1. Using the ```time-series``` program, determine the net magnetisation at 15 K and 2 T. 

2. Consult the ```hysteresis-loop``` entry in the _Vampire_ [manual](https://vampire.york.ac.uk/resources/vampire-manual.pdf) and find magnetic field needed to obtain 90% magnetisation at 15 K and 77 K. 

3. Imagine you have a $\mathrm{CrI}_3$-based device that needs at least 66% magnetisation when the field is on to run successfully. If we have a 7.5 T magnet, how hot can how can we run our device? 

In [ ]:
!cp input.CrI3 input
!/path/to/vampire-serial
!cp output output.CrI3
!cp log log.CrI3